# Phase 4: Machine Learning (Spark MLlib)
**Thực hiện bởi**: Tuấn (Group 3)

## Mục tiêu
1. **4.1** Đọc dữ liệu từ Hive `credit_transaction_db.cleaned_transactions`.
2. **4.2** `VectorAssembler` gom feature: `TX_AMOUNT`, `TX_TIME_SECONDS`, `tx_hour`, `is_weekend`, `is_night`, `customer_tx_count`, `customer_avg_amount`, `terminal_tx_count`.
3. **4.3** Class Weighting cho nhãn `TX_FRAUD` (dữ liệu lệch).
4. **4.4** Huấn luyện `LogisticRegression` (`pyspark.ml.classification`).
5. **4.5** Đánh giá bằng **Area Under PR-Curve** và **ROC-AUC** (`BinaryClassificationEvaluator`).

> **Lưu ý**: Không dùng Accuracy làm metric chính vì gian lận chiếm tỷ lệ rất nhỏ.
>
> Pipeline logic nằm trong `src/ml_module.py` (StandardScaler đặt trong pipeline ML, không sửa ETL Phase 2).

## 1. Import thư viện & module ML

In [1]:
import sys
import os

sys.path.append(os.path.abspath(".."))

from pyspark.sql import functions as F

from src.ml_module import (
    FEATURE_COLS,
    LABEL_COL,
    WEIGHT_COL,
    HIVE_TABLE,
    get_ml_spark_session,
    load_ml_dataset,
    summarize_class_balance,
    prepare_feature_frame,
    add_class_weights,
    build_ml_pipeline,
    train_and_evaluate,
    evaluate_predictions,
)

print("Feature columns:", FEATURE_COLS)
print("Label column:", LABEL_COL)
print("Hive table:", HIVE_TABLE)

Feature columns: ['TX_AMOUNT', 'TX_TIME_SECONDS', 'tx_hour', 'is_weekend', 'is_night', 'customer_tx_count', 'customer_avg_amount', 'terminal_tx_count']
Label column: TX_FRAUD
Hive table: credit_transaction_db.cleaned_transactions


## 2. Khởi tạo Spark Session (Hive Metastore / local fallback)

In [2]:
spark = get_ml_spark_session(app_name="Phase4_Machine_Learning_Tuan")
spark.sparkContext.setLogLevel("WARN")
spark

[ML] Spark session ready (Hive URI: thrift://hive-metastore:9083).


## 3. [4.1] Truy xuất dữ liệu (dataset chuẩn của project)

**Dataset chuẩn**: `simulated-data-raw-csv/` (24 tháng, có cả `TX_FRAUD = 0` và `1`).

Theo đề bài Phase 4.1, notebook đọc từ Hive bảng đã ETL:
`credit_transaction_db.cleaned_transactions`
(= bản đã làm sạch / feature engineering từ `simulated-data-raw-csv`).

Thứ tự load:
1. **Hive** (sau khi chạy ETL full) — khuyến nghị
2. Fallback CSV: **`simulated-data-raw-csv`** (không dùng `data/mock` để train)

> `data/mock` chỉ là sample test ETL của team, không phải dataset train.

In [3]:
# 4.1: ưu tiên Hive (ETL từ simulated-data-raw-csv).
# Nếu chưa có Hive → tự fallback sang folder simulated-data-raw-csv (dataset chuẩn).
df_tx, source = load_ml_dataset(spark)

print(f"Dataset source: {source}")
print(f"Total rows: {df_tx.count():,}")
df_tx.printSchema()

print("\n=== Class balance (TX_FRAUD) ===")
class_balance = summarize_class_balance(df_tx)
class_balance.show(truncate=False)

fraud_rows = [r for r in class_balance.collect() if float(r[LABEL_COL]) == 1.0]
if not fraud_rows:
    raise ValueError(
        "Không thấy TX_FRAUD=1. Đang không dùng đúng dataset chuẩn. "
        "Hãy ETL từ simulated-data-raw-csv rồi chạy lại cell này."
    )
print(f"OK — có {int(fraud_rows[0]['count']):,} giao dịch gian lận trong dataset.")

[ML] Loaded 14,158,971 rows from Hive table `credit_transaction_db.cleaned_transactions`.
Dataset source: hive
Total rows: 14,158,971
root
 |-- TRANSACTION_ID: long (nullable = true)
 |-- TX_DATETIME: timestamp (nullable = true)
 |-- CUSTOMER_ID: long (nullable = true)
 |-- TERMINAL_ID: long (nullable = true)
 |-- TX_AMOUNT: double (nullable = true)
 |-- TX_TIME_SECONDS: long (nullable = true)
 |-- TX_TIME_DAYS: integer (nullable = true)
 |-- TX_FRAUD: integer (nullable = true)
 |-- TX_FRAUD_SCENARIO: integer (nullable = true)
 |-- tx_date: date (nullable = true)
 |-- tx_hour: integer (nullable = true)
 |-- tx_day_of_week: string (nullable = true)
 |-- is_weekend: integer (nullable = true)
 |-- is_night: integer (nullable = true)
 |-- customer_tx_count: long (nullable = true)
 |-- customer_avg_amount: double (nullable = true)
 |-- terminal_tx_count: long (nullable = true)
 |-- tx_year_month: string (nullable = true)


=== Class balance (TX_FRAUD) ===
+--------+--------+--------+
|TX_FR

## 4. [4.2] Feature Engineering — VectorAssembler
- Chỉ giữ các cột feature được phân công.
- **Loại bỏ** định danh: `TRANSACTION_ID`, `CUSTOMER_ID`, `TERMINAL_ID`.
- **Không dùng** `TX_FRAUD_SCENARIO` (tránh label leakage).
- Pipeline: `VectorAssembler` → `StandardScaler` → `LogisticRegression`.

In [4]:
df_features = prepare_feature_frame(df_tx)
print("Prepared columns:", df_features.columns)
df_features.show(5, truncate=False)

pipeline = build_ml_pipeline()
for idx, stage in enumerate(pipeline.getStages()):
    print(f"Stage {idx}: {stage.__class__.__name__}")

Prepared columns: ['TX_AMOUNT', 'TX_TIME_SECONDS', 'tx_hour', 'is_weekend', 'is_night', 'customer_tx_count', 'customer_avg_amount', 'terminal_tx_count', 'TX_FRAUD']
+---------+---------------+-------+----------+--------+-----------------+-------------------+-----------------+--------+
|TX_AMOUNT|TX_TIME_SECONDS|tx_hour|is_weekend|is_night|customer_tx_count|customer_avg_amount|terminal_tx_count|TX_FRAUD|
+---------+---------------+-------+----------+--------+-----------------+-------------------+-----------------+--------+
|92.12    |3.145435E7     |1.0    |1.0       |1.0     |54.0             |58.21000000000001  |4.0              |0.0     |
|43.65    |3.0975889E7    |12.0   |0.0       |0.0     |99.0             |49.02959595959595  |8.0              |0.0     |
|40.0     |2.9658711E7    |6.0    |1.0       |0.0     |117.0            |68.01478632478634  |4.0              |0.0     |
|42.37    |3.0189975E7    |10.0   |1.0       |0.0     |90.0             |29.27233333333334  |8.0             

## 5. [4.3] Class Weighting
Công thức balanced: `weight_c = n / (n_classes * n_c)`.

Class gian lận (thiểu số) nhận trọng số cao hơn → Logistic Regression không bỏ qua nhãn hiếm.

In [5]:
df_weighted = add_class_weights(df_features)

df_weighted.groupBy(LABEL_COL, WEIGHT_COL).count().orderBy(LABEL_COL).show(truncate=False)
df_weighted.select(*FEATURE_COLS[:3], LABEL_COL, WEIGHT_COL).show(8, truncate=False)

[ML] Class counts: {1.0: 35564, 0.0: 14123407}
[ML] Class weights: {1.0: 199.06325216511078, 0.0: 0.5012590446483628}
+--------+------------------+--------+
|TX_FRAUD|class_weight      |count   |
+--------+------------------+--------+
|0.0     |0.5012590446483628|14123407|
|1.0     |199.06325216511078|35564   |
+--------+------------------+--------+

+---------+---------------+-------+--------+------------------+
|TX_AMOUNT|TX_TIME_SECONDS|tx_hour|TX_FRAUD|class_weight      |
+---------+---------------+-------+--------+------------------+
|92.12    |3.145435E7     |1.0    |0.0     |0.5012590446483628|
|43.65    |3.0975889E7    |12.0   |0.0     |0.5012590446483628|
|40.0     |2.9658711E7    |6.0    |0.0     |0.5012590446483628|
|42.37    |3.0189975E7    |10.0   |0.0     |0.5012590446483628|
|32.32    |3.1314144E7    |10.0   |0.0     |0.5012590446483628|
|95.23    |3.0280671E7    |11.0   |0.0     |0.5012590446483628|
|50.99    |3.0464375E7    |14.0   |0.0     |0.5012590446483628|
|111.34

## 6. [4.4] Train / Test split & huấn luyện Logistic Regression
- Split 80/20, `seed=42`.
- `LogisticRegression` dùng `weightCol=class_weight`.

In [6]:
model, metrics, train_df, predictions = train_and_evaluate(
    df_tx,
    train_ratio=0.8,
    seed=42,
)

print("Train class balance:")
train_df.groupBy(LABEL_COL).count().show()

print("Sample predictions (probability / prediction):")
predictions.select(
    LABEL_COL,
    "prediction",
    "probability",
).show(10, truncate=False)

[ML] Class counts: {1.0: 35564, 0.0: 14123407}
[ML] Class weights: {1.0: 199.06325216511078, 0.0: 0.5012590446483628}
[ML] Train rows=11,328,146 | Test rows=2,830,825
[ML] Evaluation on test set -> Area Under PR = 0.468655 | ROC-AUC = 0.806468
[ML] Note: Accuracy is intentionally not used (severe class imbalance).
Train class balance:
+--------+--------+
|TX_FRAUD|   count|
+--------+--------+
|     1.0|   28417|
|     0.0|11299729|
+--------+--------+

Sample predictions (probability / prediction):
+--------+----------+----------------------------------------+
|TX_FRAUD|prediction|probability                             |
+--------+----------+----------------------------------------+
|0.0     |0.0       |[0.737021783699467,0.262978216300533]   |
|0.0     |0.0       |[0.7167713980712185,0.2832286019287815] |
|0.0     |0.0       |[0.7335280335774415,0.26647196642255855]|
|0.0     |0.0       |[0.7328319632608418,0.26716803673915823]|
|0.0     |0.0       |[0.7198671372294821,0.28013286277

## 7. [4.5] Đánh giá — Area Under PR & ROC-AUC
Dùng `BinaryClassificationEvaluator`:
- `metricName="areaUnderPR"` (ưu tiên khi data lệch)
- `metricName="areaUnderROC"`

In [7]:
metrics_check = evaluate_predictions(predictions)

print("=" * 60)
print("PHASE 4 EVALUATION RESULTS (Test Set)")
print("=" * 60)
print(f"Area Under PR-Curve : {metrics_check['areaUnderPR']:.6f}")
print(f"ROC-AUC             : {metrics_check['areaUnderROC']:.6f}")
print("=" * 60)
print("Accuracy intentionally omitted (severe class imbalance).")

# Confusion-style counts (support slide/report; not used as primary metric)
predictions.groupBy(LABEL_COL, "prediction").count().orderBy(LABEL_COL, "prediction").show()

PHASE 4 EVALUATION RESULTS (Test Set)
Area Under PR-Curve : 0.468655
ROC-AUC             : 0.806466
Accuracy intentionally omitted (severe class imbalance).
+--------+----------+-------+
|TX_FRAUD|prediction|  count|
+--------+----------+-------+
|     0.0|       0.0|2496105|
|     0.0|       1.0| 327573|
|     1.0|       0.0|   2603|
|     1.0|       1.0|   4544|
+--------+----------+-------+



## 8. Kết luận Phase 4
- Đã hoàn thành pipeline Spark MLlib: **Hive → VectorAssembler (+ StandardScaler) → Class Weighting → Logistic Regression → PR-AUC / ROC-AUC**.
- Module tái sử dụng: `src/ml_module.py`.
- Chạy lại full production path:
  1. `docker compose -f docker/docker-compose.yml up -d`
  2. ETL full data: `docker exec jupyter-lab spark-submit --driver-memory 4g /home/jovyan/src/etl_module_credit_transaction.py data/simulated-data-raw-csv`
  3. Run All notebook này (hoặc `spark-submit .../src/ml_module.py`).